In [ ]:
!pip install ultralytics --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.2 MB/s eta 0:00:00


#YOLOv26

# Subset: Components Only (PCB-MC-C)


In [1]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/results/YOLOV26/components_only")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FAMILY = "yolo26"   # "yolov8" | "yolo11" | "yolo26"
SIZE = "s"                # "n" | "s" | "m" | "l" | "x" (as available)
EPOCHS = 300

# A100-friendly
IMGSZ = 1024
BATCH = -1               # auto-batch (great on A100)
WORKERS = 8
CACHE = True
AMP = True

# Optimizer / schedule
OPTIMIZER = "AdamW"
COS_LR = True
PATIENCE = 25

# Base LR (tuned per family below)
LR0_V8  = 1e-4
LR0_V11 = 3e-4
LR0_V26 = 2.5e-4

# PCB-safe augment defaults (tuned per family below)
BASE_AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    copy_paste=0.0,
    close_mosaic=20,   # disable mosaic near the end
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}.")
    if "nc" not in data:
        data["nc"] = len(data["names"])
    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")
    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]
    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")
    return data

def weights_name(family: str, size: str) -> str:
    if family == "yolov8":
        return f"yolov8{size}.pt"
    if family == "yolo11":
        return f"yolo11{size}.pt"
    if family == "yolo26":
        return f"yolo26{size}.pt"
    raise ValueError("MODEL_FAMILY must be: yolov8 | yolo11 | yolo26")

def family_hypers(family: str):
    aug = dict(BASE_AUG)

    if family == "yolov8":
        lr0 = LR0_V8
        aug.update(mosaic=0.6, mixup=0.05)
    elif family == "yolo11":
        lr0 = LR0_V11
        aug.update(mosaic=0.4, mixup=0.02)
    elif family == "yolo26":
        lr0 = LR0_V26
        # More conservative mixing for structured PCB scenes
        aug.update(mosaic=0.2, mixup=0.0)
    else:
        raise ValueError("Unknown family")

    return lr0, aug

BASE_WEIGHTS = weights_name(MODEL_FAMILY, SIZE)
LR0, AUG = family_hypers(MODEL_FAMILY)

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"
    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | {MODEL_FAMILY}{SIZE}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH} | lr0: {LR0}")
    print(f"  YAML: {yaml_path} | nc: {cfg['nc']}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH / MODEL_FAMILY),
        name=f"{MODEL_FAMILY}{SIZE}_fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        amp=AMP,
        cache=CACHE,
        workers=WORKERS,

        **AUG
    )

print("\n✅ Training complete for all folds!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Training Fold 0 | yolo26s
  weights: yolo26s.pt | imgsz: 1024 | batch: -1 | lr0: 0.00025
  YAML: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml | nc: 23
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data

# Subset: Full Dataset (PCB-MC-A)

In [2]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FAMILY = "yolo26"   # "yolov8" | "yolo11" | "yolo26"
SIZE = "s"                # "n" | "s" | "m" | "l" | "x" (as available)
EPOCHS = 300

# A100-friendly
IMGSZ = 1024
BATCH = -1               # auto-batch (great on A100)
WORKERS = 8
CACHE = True
AMP = True

# Optimizer / schedule
OPTIMIZER = "AdamW"
COS_LR = True
PATIENCE = 25

# Base LR (tuned per family below)
LR0_V8  = 1e-4
LR0_V11 = 3e-4
LR0_V26 = 2.5e-4

# PCB-safe augment defaults (tuned per family below)
BASE_AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    copy_paste=0.0,
    close_mosaic=20,   # disable mosaic near the end
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}.")
    if "nc" not in data:
        data["nc"] = len(data["names"])
    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")
    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]
    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")
    return data

def weights_name(family: str, size: str) -> str:
    if family == "yolov8":
        return f"yolov8{size}.pt"
    if family == "yolo11":
        return f"yolo11{size}.pt"
    if family == "yolo26":
        return f"yolo26{size}.pt"
    raise ValueError("MODEL_FAMILY must be: yolov8 | yolo11 | yolo26")

def family_hypers(family: str):
    aug = dict(BASE_AUG)

    if family == "yolov8":
        lr0 = LR0_V8
        aug.update(mosaic=0.6, mixup=0.05)
    elif family == "yolo11":
        lr0 = LR0_V11
        aug.update(mosaic=0.4, mixup=0.02)
    elif family == "yolo26":
        lr0 = LR0_V26
        # More conservative mixing for structured PCB scenes
        aug.update(mosaic=0.2, mixup=0.0)
    else:
        raise ValueError("Unknown family")

    return lr0, aug

BASE_WEIGHTS = weights_name(MODEL_FAMILY, SIZE)
LR0, AUG = family_hypers(MODEL_FAMILY)

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"
    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | {MODEL_FAMILY}{SIZE}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH} | lr0: {LR0}")
    print(f"  YAML: {yaml_path} | nc: {cfg['nc']}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH / MODEL_FAMILY),
        name=f"{MODEL_FAMILY}{SIZE}_fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        amp=AMP,
        cache=CACHE,
        workers=WORKERS,

        **AUG
    )

print("\n✅ Training complete for all folds!")


🚀 Training Fold 0 | yolo26s
  weights: yolo26s.pt | imgsz: 1024 | batch: -1 | lr0: 0.00025
  YAML: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml | nc: 31
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=Non

# Subset: Missing Only (PCB-MC-M)

In [ ]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FAMILY = "yolo26"   # "yolov8" | "yolo11" | "yolo26"
SIZE = "s"                # "n" | "s" | "m" | "l" | "x" (as available)
EPOCHS = 300

# A100-friendly
IMGSZ = 1024
BATCH = -1               # auto-batch (great on A100)
WORKERS = 8
CACHE = True
AMP = True

# Optimizer / schedule
OPTIMIZER = "AdamW"
COS_LR = True
PATIENCE = 25

# Base LR (tuned per family below)
LR0_V8  = 1e-4
LR0_V11 = 3e-4
LR0_V26 = 2.5e-4

# PCB-safe augment defaults (tuned per family below)
BASE_AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    copy_paste=0.0,
    close_mosaic=20,   # disable mosaic near the end
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}.")
    if "nc" not in data:
        data["nc"] = len(data["names"])
    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")
    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]
    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")
    return data

def weights_name(family: str, size: str) -> str:
    if family == "yolov8":
        return f"yolov8{size}.pt"
    if family == "yolo11":
        return f"yolo11{size}.pt"
    if family == "yolo26":
        return f"yolo26{size}.pt"
    raise ValueError("MODEL_FAMILY must be: yolov8 | yolo11 | yolo26")

def family_hypers(family: str):
    aug = dict(BASE_AUG)

    if family == "yolov8":
        lr0 = LR0_V8
        aug.update(mosaic=0.6, mixup=0.05)
    elif family == "yolo11":
        lr0 = LR0_V11
        aug.update(mosaic=0.4, mixup=0.02)
    elif family == "yolo26":
        lr0 = LR0_V26
        # More conservative mixing for structured PCB scenes
        aug.update(mosaic=0.2, mixup=0.0)
    else:
        raise ValueError("Unknown family")

    return lr0, aug

BASE_WEIGHTS = weights_name(MODEL_FAMILY, SIZE)
LR0, AUG = family_hypers(MODEL_FAMILY)

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"
    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | {MODEL_FAMILY}{SIZE}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH} | lr0: {LR0}")
    print(f"  YAML: {yaml_path} | nc: {cfg['nc']}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH / MODEL_FAMILY),
        name=f"{MODEL_FAMILY}{SIZE}_fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        amp=AMP,
        cache=CACHE,
        workers=WORKERS,

        **AUG
    )

print("\n✅ Training complete for all folds!")


🚀 Training Fold 0 | yolo26s
  weights: yolo26s.pt | imgsz: 1024 | batch: -1 | lr0: 0.00025
  YAML: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml | nc: 8
Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002

# Subset: Non-missing (PCB-MC-P)

In [ ]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FAMILY = "yolo26"   # "yolov8" | "yolo11" | "yolo26"
SIZE = "s"                # "n" | "s" | "m" | "l" | "x" (as available)
EPOCHS = 300

# A100-friendly
IMGSZ = 1024
BATCH = -1               # auto-batch (great on A100)
WORKERS = 8
CACHE = True
AMP = True

# Optimizer / schedule
OPTIMIZER = "AdamW"
COS_LR = True
PATIENCE = 25

# Base LR (tuned per family below)
LR0_V8  = 1e-4
LR0_V11 = 3e-4
LR0_V26 = 2.5e-4

# PCB-safe augment defaults (tuned per family below)
BASE_AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    copy_paste=0.0,
    close_mosaic=20,   # disable mosaic near the end
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}.")
    if "nc" not in data:
        data["nc"] = len(data["names"])
    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")
    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]
    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")
    return data

def weights_name(family: str, size: str) -> str:
    if family == "yolov8":
        return f"yolov8{size}.pt"
    if family == "yolo11":
        return f"yolo11{size}.pt"
    if family == "yolo26":
        return f"yolo26{size}.pt"
    raise ValueError("MODEL_FAMILY must be: yolov8 | yolo11 | yolo26")

def family_hypers(family: str):
    aug = dict(BASE_AUG)

    if family == "yolov8":
        lr0 = LR0_V8
        aug.update(mosaic=0.6, mixup=0.05)
    elif family == "yolo11":
        lr0 = LR0_V11
        aug.update(mosaic=0.4, mixup=0.02)
    elif family == "yolo26":
        lr0 = LR0_V26
        # More conservative mixing for structured PCB scenes
        aug.update(mosaic=0.2, mixup=0.0)
    else:
        raise ValueError("Unknown family")

    return lr0, aug

BASE_WEIGHTS = weights_name(MODEL_FAMILY, SIZE)
LR0, AUG = family_hypers(MODEL_FAMILY)

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"
    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | {MODEL_FAMILY}{SIZE}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH} | lr0: {LR0}")
    print(f"  YAML: {yaml_path} | nc: {cfg['nc']}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH / MODEL_FAMILY),
        name=f"{MODEL_FAMILY}{SIZE}_fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        amp=AMP,
        cache=CACHE,
        workers=WORKERS,

        **AUG
    )

print("\n✅ Training complete for all folds!")


🚀 Training Fold 0 | yolo26s
  weights: yolo26s.pt | imgsz: 1024 | batch: -1 | lr0: 0.00025
  YAML: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml | nc: 23
Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00025